The goal of this notebook is to analyse and aggregate the bureau table, combine it with the bureau_balance table, and use the derived information to enhance predictions in the application.

# Setup

In [1]:
from helpers import preprocessing
import pandas as pd

In [6]:
bureau = preprocessing.load_pkl_to_preprocessor('bureau')
display(bureau.convert_all('category').info(show_counts=True))
print(bureau.column_structure)
bureau_df = bureau.data

Loading tables: ['bureau.csv']
loaded bureau with shape (1716428, 17)
dict_keys(['credit_status', 'bureau_overdue', 'time_variables', 'bureau', 'bureau_balance', 'pos_cash_balance', 'balance_limits', 'drawings_payments', 'contract_installments', 'credit_card_balance', 'previous_application', 'installments_payments'])
No missing columns.
Unexpected columns: ['SK_ID_CURR', 'SK_ID_BUREAU']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Non-Null Count    Dtype   
---  ------                  --------------    -----   
 0   SK_ID_CURR              1716428 non-null  int64   
 1   SK_ID_BUREAU            1716428 non-null  int64   
 2   CREDIT_ACTIVE           1716428 non-null  category
 3   CREDIT_CURRENCY         1716428 non-null  category
 4   DAYS_CREDIT             1716428 non-null  int64   
 5   CREDIT_DAY_OVERDUE      1716428 non-null  int64   
 6   DAYS_CREDIT_ENDDATE     1610875 non-null  fl

None

['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']


# EDA

### Bureau Description:
bureau.csv

* All client's previous credits provided by other financial institutions that were reported to Credit Bureau (for clients who have a loan in our sample).
* For every loan in our sample, there are as many rows as number of credits the client had in Credit Bureau before the application date.



#### unique applicants

The application train and test tables are 307511 and 48744 values, respectively. For a total of 356255

In [8]:
print("Unique clients in bureau:", bureau_df["SK_ID_CURR"].unique().shape[0])
print(
    "Difference from application train and test:",
    bureau_df["SK_ID_CURR"].unique().shape[0] - 356255,
)

Unique clients in bureau: 305811
Difference from application train and test: -50444


A significant portion of applicants have no known history of previous loans within the bureau. NaN robust techniques or imputation are required.

bureau

## Supplementations of applications by state.
We include the last 5 stages of information to supplement the application table. The heuristic is that most recent data will be most relevant for individuals status.

In [ ]:
balance_5 = balance.data[balance.data["MONTHS_BALANCE"] >= -5].copy()
balance_5 = balance_5.pivot_table(
    index="SK_ID_BUREAU", columns="MONTHS_BALANCE", values="STATUS", aggfunc="first"
)
balance_5.reset_index(inplace=True)
balance_5 = balance_5.merge(
    balance.data.groupby("SK_ID_BUREAU")["MONTHS_BALANCE"].min(),
    on="SK_ID_BUREAU",
    how="left",
)
balance_5.head(5)

,SK_ID_BUREAU,-5,-4,-3,-2,-1,0,MONTHS_BALANCE
0,5001709,C,C,C,C,C,C,-96
1,5001710,C,C,C,C,C,C,-82
2,5001711,NaN,NaN,0,0,0,X,-3
3,5001712,C,C,C,C,C,C,-18
4,5001713,X,X,X,X,X,X,-21


In [ ]:
import importlib

from sklearn.tests.test_min_dependencies_readme import extra
importlib.reload(preprocessing)
app = preprocessing.load_pkl_to_preprocessor('application_test')
app.convert_all('category')
app = app.get_expected_df(extra=['SK_ID_BUREAU'])
print(app.columns)
app = app.merge(balance_5, on='SK_ID_BUREAU', how='left')

NameError: name 'importlib' is not defined